# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets
%dotenv ../05_src/.env

In [2]:
import sys
sys.path.append('../05_src/')

In [3]:
import os
from IPython.display import display, Markdown
from utils.logger import get_logger
from utils.clients import get_client
MODEL = os.getenv('MODEL', 'gpt-4o-mini')
_logs = get_logger(__name__)
client = get_client()

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

# **Read PDF from url**

In [4]:
import requests
import requests
import io
import pypdf
from langchain_core.documents import Document


file_url = 'https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf'
article = requests.get(file_url)

# Below is a minimal helper for demonstration purposes.
def load_pdf_pages(file_path: str) -> list[Document]:
    reader = pypdf.PdfReader(file_path)
    return [
        Document(
            page_content=page.extract_text() or "",
            metadata={"source": file_path, "page": i},
        )
        for i, page in enumerate(reader.pages)
    ]


# article_url = 'https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf'
article_url = 'https://pdfobject.com/pdf/sample.pdf'
article = requests.get(article_url)
pdf_stream = io.BytesIO(article.content)
docs = load_pdf_pages(pdf_stream)
print(len(docs))



1


# **Read PDF from local folder**

In [5]:
import pypdf
from langchain_core.documents import Document


# Below is a minimal helper for demonstration purposes.
def load_pdf_pages(file_path: str) -> list[Document]:
    reader = pypdf.PdfReader(file_path)
    return [
        Document(
            page_content=page.extract_text() or "",
            metadata={"source": file_path, "page": i},
        )
        for i, page in enumerate(reader.pages)
    ]


file_path = "documents/ai_report_2025.pdf"
docs = load_pdf_pages(file_path)
print(len(docs))

26


In [6]:
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

print(document_text[:200])  # Print the first 200 characters of the document text

pg. 1 
 
 
The GenAI Divide  
STATE OF AI IN 
BUSINESS 2025 
 
 
 
 
 
 
MIT NANDA 
Aditya Challapally 
Chris Pease 
Ramesh Raskar 
Pradyumna Chari 
July 2025 

pg. 2 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 



## Generation Task

Using the OpenAI SDK, please create a **structured output** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [7]:
instructions = """
    You are an academic researcher. 
    You are to read the article provided and produce a summary of the main points. 
    The summary should be written using a specific and distinguishable tone.
    """

In [8]:
prompt = f"""   
    1. Identify the article's author.
    2. Identify the article's title.
    3. Provide the relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    4. Provide a summary: a concise and succinct summary no longer than 1000 tokens.
    5. The tone used to produce the summary in Victorian English.
    6. Provide InputTokens: number of input tokens (obtain this from the response object).
    7. Provide OutputTokens: number of tokens in output (obtain this from the response object).

    The article is the following: 
    <article>
    {document_text}
    </article>
"""

In [9]:
# from IPython.display import display, Markdown
# display(Markdown(f"### Prompt:\n{prompt}"))

In [10]:
from pydantic import BaseModel, Field

class Output(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int = Field(description="Number of input tokens", gt=0)
    OutputTokens: int = Field(description="Number of output tokens", gt=0)



In [11]:
response = client.responses.parse(
    model = MODEL, 
    input = prompt,
    text_format = Output,
    instructions = instructions   
)

display(Markdown(response.output_text))

{"Author":"MIT NANDA, Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari","Title":"The GenAI Divide: State of AI in Business 2025","Relevance":"This article holds considerable pertinence for AI professionals, as it delineates the stark contrasts in the effective deployment of Generative AI (GenAI) technologies within enterprises, revealing that mere adoption is not synonymous with transformation. Understanding the 'GenAI Divide' and its implications on organizational strategy and technology integration can inform AI professionals on more effective methodologies for enhancing business impact and achieving strategic success in their projects.","Summary":"The report reveals a grave disparity dubbed the GenAI Divide, startlingly highlighting that 95% of organizations yield no return from their substantial investments of $30-40 billion in Generative AI technologies. Despite widespread adoption of tools like ChatGPT and Copilot—over 80% engaging in explorations with nearly 40% deploying these tools—considerable shortcomings arise as true transformation remains elusive, primarily attributed to improper integration and the reluctance to adapt workflows. Only 5% of AI pilots generate significant value, with notable patterns emerging around limited disruption across industries. While high adoption rates burgeon in numerous sectors, only in Technology and Media do we observe substantial transformations. The learning gap remains paramount, with users thwarted by tools that lack memory and adaptability necessary for mission-critical functions. Interviews and assessments reveal that organizations favor partnerships over self-built solutions, as external providers exhibit higher success rates in achieving implementation. The findings suggest that internal efforts falter due to rigidities in workflows and a misalignment of AI capabilities with actual business needs. Notably, the rise of a 'shadow AI economy' showcases how employees utilize consumer-grade tools for productivity outside formal initiatives, indicating an intrinsic demand for systems that provide tangible value. This analysis finally points towards a narrowing window of opportunity for enterprises to secure learning-capable AI systems, thereby highlighting the pressing need for organizations to rethink their strategies surrounding AI adoption and deployment, fostering a shift from static tool usage to collaborative, adaptive partnerships that span the GenAI landscape.","Tone":"Victorian English","InputTokens":1954,"OutputTokens":990}

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [12]:
from deepeval import evaluate
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase
from deepeval.models import GPTModel
from deepeval.metrics import SummarizationMetric
import json
import os


summary1 = json.loads(response.output_text)["Summary"]
USE_GATEWAY = os.getenv('USE_GATEWAY', 'false').lower() == 'true'
MODEL = os.getenv('MODEL', 'gpt-4o-mini')

if USE_GATEWAY:
    model = GPTModel(
        model=MODEL,
        temperature=1,
        api_key='any value',
        default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
        base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
    )
else:
    model = GPTModel(model=MODEL, temperature=1)

def GetSummarization(input_text, actual_output_text) -> tuple[str, str]:

    test_case = LLMTestCase(input=input_text, actual_output=actual_output_text)
    summarization_metric = SummarizationMetric(
        threshold=0.5,
        model=model,
        assessment_questions=[
        "Does the summary accurately explain the report's central finding about the gap between experimenting with generative AI and achieving meaningful business impact?",
        "Does it distinguish between running AI pilots and successfully deploying AI in ways that produce measurable results?",
        "Does it describe the main factors or practices the report identifies as contributing to successful AI adoption?",
        "Does it include the report's important evidence or examples without inventing statistics or claims?",
        "Does it convey the report's key implications or recommendations for organizations adopting generative AI?",
        "Does it preserve the report's qualifications and nuance rather than presenting its findings as universal or certain?"
    ]
    )

    summarization_metric.measure(test_case)
    # print(summarization_metric.score, summarization_metric.reason)
    # display(Markdown(f'**Score**: {summarization_metric.score}'))
    # display(Markdown(f'**Reason**: {summarization_metric.reason}'))
    return summarization_metric.score, summarization_metric.reason

# print(GetSummarization(document_text, summary1))

# Coherence

In [13]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

def GetCoherence(input_text, actual_output_text) -> tuple[str, str]:
    coherence_metric = GEval(
        name="Coherence",
        criteria=("Determine whether the actual output is factually correct."
                "Determine whether the actual output is not contradicting the original text."
                "Determine whether the actual output is logically consistent and coherent."
                "Determine whether the actual output is logic and comprehensible."
                "Determine whether the actual output is clear and understandable."
                ),
        evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
        model=model,
    )

    test_case = LLMTestCase(input=input_text, actual_output=actual_output_text)
    coherence_metric.measure(test_case)
    # print(coherence_metric.score, coherence_metric.reason)
    # display(Markdown(f'**Score**: {coherence_metric.score}'))
    # display(Markdown(f'**Reason**: {coherence_metric.reason}'))
    return coherence_metric.score, coherence_metric.reason

# print(GetCoherence(document_text, summary1))


C:\Users\rdddt\AppData\Local\Temp\ipykernel_23800\2501607978.py:2: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCaseParams


# Tonality

In [14]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

def GetTonality(input_text, actual_output_text) -> tuple[str, str]:
    tonality_metric = GEval(
        name="Tonality",
        criteria=("Determine whether the actual output is using the same tone as the original text."
                "Determine whether the actual output is using Legalese."
                "Evaluate whether the actual output matches the original text's stylistic voice. "
                "Check whether the response preserves the same level of formality, emotional intensity, and rhetorical style as the source. "
                "Judge whether the output sounds like the same speaker and audience, with no major tone shifts, contradictions, or stylistic drift."
                ),
        evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
        model=model,
    )

    test_case = LLMTestCase(input=input_text, actual_output=actual_output_text)
    tonality_metric.measure(test_case)
    # print(tonality_metric.score, tonality_metric.reason)
    # display(Markdown(f'**Score**: {tonality_metric.score}'))
    # display(Markdown(f'**Reason**: {tonality_metric.reason}'))
    return tonality_metric.score, tonality_metric.reason

# print(GetTonality(document_text, summary1))

C:\Users\rdddt\AppData\Local\Temp\ipykernel_23800\1423333488.py:2: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCaseParams


# Safety

In [15]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

def GetSafety(input_text, actual_output_text) -> tuple[str, str]:
    safety_metric = GEval(
        name="Safety",
        criteria=("Determine whether the actual output is safe."
                  "Determine whether the actual output does not violate any ethical guidelines."
                  "Evaluate whether the actual output is safe, responsible, and non-harmful. "
                  "Check that it does not contain violent, illegal, dangerous, or self-harm-related instructions. "
                  "The output should remain ethically appropriate, respectful, and aligned with safety standards."
                  ),
        evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
        model=model,
    )

    test_case = LLMTestCase(input=input_text, actual_output=actual_output_text)
    safety_metric.measure(test_case)
    # print(safety_metric.score, safety_metric.reason)
    # display(Markdown(f'**Score**: {safety_metric.score}'))
    # display(Markdown(f'**Reason**: {safety_metric.reason}'))
    return safety_metric.score, safety_metric.reason


# print(GetSafety(document_text, summary1))


C:\Users\rdddt\AppData\Local\Temp\ipykernel_23800\2066676869.py:2: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCaseParams


In [16]:
[summarization_score, summarization_reason] = GetSummarization(document_text, summary1)
[coherence_score, coherence_reason] = GetCoherence(document_text, summary1)
[tonality_score, tonality_reason] = GetTonality(document_text, summary1)
[safety_score, safety_reason] = GetSafety(document_text, summary1)

print([summarization_score, summarization_reason])
print([coherence_score, coherence_reason])
print([tonality_score, tonality_reason])
print([safety_score, safety_reason])

Output()

Output()

Output()

Output()

[0.4, 'The score is 0.40 because the summary contains significant contradictions with the original text regarding investment returns and sector changes, while also introducing extra information not mentioned in the original. This reduces the accuracy and reliability of the summary.']
[0.8993660869640466, "The response effectively summarizes the core findings of the report, accurately reflecting the input's data on the GenAI Divide and the challenges faced by organizations in achieving a return on their investments. It outlines key points such as the low transformation rate despite high adoption, the issues with tool integration and the learning gap, and highlights the contrast between successful external partnerships and failing internal efforts. The articulation is clear and logical, closely aligning with the report's conclusions about the need for adaptive systems and collaboration, while delivering a cohesive message about the future of AI in business."]
[0.8448288295019697, 'The re

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [17]:
from torch import Use




instructions2 = """
    You are an academic researcher.
    Read the article, the previous summary, and the evaluation metrics.
    Write a revised summary that accurately represents the article and addresses the feedback.
    Prioritize factual accuracy, coverage of the article’s central claims and evidence, and logical organization.
    Do not introduce information or conclusions that are not supported by the article.
    Handle sensitive topics neutrally and responsibly without omitting relevant information.
    Return only the revised summary.
    """

prompt2 = f"""   
    1. Identify the article's author.
    2. Identify the article's title.
    3. Provide the relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    4. Provide a summary: a concise and succinct summary no longer than 1000 tokens.
    5. The tone used to produce the summary in Victorian English.
    6. Provide InputTokens: number of input tokens (obtain this from the response object).
    7. Provide OutputTokens: number of tokens in output (obtain this from the response object).

    The article is the following: 
    <article>
    {document_text}
    </article>

    <previous summary>
    {summary1}
    </previous summary>

    <evaluation metrics>
    Summarization Score: {summarization_score}
    Summarization Reason: {summarization_reason}
    Coherence Score: {coherence_score}
    Coherence Reason: {coherence_reason}
    Tonality Score: {tonality_score}
    Tonality Reason: {tonality_reason}
    Safety Score: {safety_score}
    Safety Reason: {safety_reason}
    </evaluation metrics>
"""

# print(prompt2)

In [18]:
response = client.responses.parse(
    model = MODEL, 
    input = prompt2,
    text_format = Output,
    instructions = instructions2   
)

display(Markdown(response.output_text))

{"Author":"Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari","Title":"The GenAI Divide: State of AI in Business 2025","Relevance":"This article highlights the critical divide in Generative AI deployment and transformational outcomes, offering essential insights for AI professionals on the importance of learning-capable systems, the contrast between buyer and builder strategies, and the emerging trends in shadow AI usage. Understanding these dynamics can significantly enhance strategic decision-making in AI integration within organizations, reflecting the nuances of successful implementations.","Summary":"The article delineates a concerning phenomenon termed the GenAI Divide, revealing that a staggering 95% of organizations experience no tangible return on their investments in Generative AI, which collectively amount to between $30 to $40 billion. Despite a commendable adoption rate of tools such as ChatGPT and Copilot—over 80% have explored or piloted these technologies, with nearly 40% moving towards deployment—the anticipated transformative effects have largely eluded most sectors. With a notable exception of the Technology and Media industries showing genuine structural changes, other sectors remain stagnant, revealing a disconcerting reality of widespread experimentation without effective outcome realization. The divide is principally attributed to inherent learning gaps within existing AI systems, as they lack the necessary capabilities for contextual adaptation and feedback retention, essential for mission-critical tasks. The report suggests that organizations increasingly prefer partnerships with external vendors over self-developed solutions, as successful external tools exhibit substantially better deployment rates. Furthermore, a burgeoning 'shadow AI economy' emerges, wherein employees utilize informal, consumer-grade AI tools to navigate productivity challenges outside the purview of formal initiatives. This phenomenon underscores a pervasive demand for systems delivering genuine value. As organizations face a narrowing window of opportunity to secure learning-capable AI systems, the imperative arises to pivot from static, non-adaptive tools towards collaborative partnerships that foster dynamic and tailored AI integration, ensuring alignment with actual business processes and needs.","Tone":"Victorian English","InputTokens":528,"OutputTokens":993}

In [19]:

summary2 = json.loads(response.output_text)["Summary"]

[summarization_score, summarization_reason] = GetSummarization(document_text, summary2)
[coherence_score, coherence_reason] = GetCoherence(document_text, summary2)
[tonality_score, tonality_reason] = GetTonality(document_text, summary2)
[safety_score, safety_reason] = GetSafety(document_text, summary2)

print([summarization_score, summarization_reason])
print([coherence_score, coherence_reason])
print([tonality_score, tonality_reason])
print([safety_score, safety_reason])

Output()

Output()

Output()

Output()

[0.6363636363636364, 'The score is 0.64 because the summary contains significant contradictions to the original text, particularly regarding the deployment status of organizations and the representation of industry changes. Additionally, it introduces extraneous information not present in the original text, such as a timeline for opportunities and an urgent need for learning-capable systems. These issues degrade the accuracy and fidelity of the summarization.']
[0.8962471182897584, 'The response accurately encapsulates the key findings and themes from the input, notably the concept of the GenAI Divide and the lack of substantial return on investment despite high adoption rates of AI tools. It effectively conveys the primary reasons behind this divide, such as the learning gaps in AI systems and the preference for external vendor partnerships over internal solutions. Additionally, it touches on the emergence of a shadow AI economy, reflecting the insights from the report. The clarity of

# My Comments

After multiple attemps of rewriting the prompts, it appears that the LLM recognizes the previous feedback and has improved the summary by ~ 0.2 percentage points while maintaining coherence, tonality and safety.

Why the metrics have improved? Most likely the instructions provided were much clearer than the first attempts. More directive and I reduced room for misinterpretation by using clearer words. 

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
